In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

# ------------------- PATHS: ADJUST TO YOUR CAIR LAYOUT -------------------
CSV_CLEAN = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/data/processed/tft_clean.csv"
CSV_LEADS = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/data/raw/openmeteo_lead_forecasts.csv"
CKPT      = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/checkpoints/tft_clean_best.ckpt"
OUT_JSON  = "metrics_lead_matched.json"
# --------------------------------------------------------------------------

ENCODER_LEN, DECODER_LEN = 168, 120
VAL_START = "2023-01-01"
BATCH = 256
BASELINE = [3.84, 4.07, 4.12, 4.14, 4.16]

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "apparent_temperature", "precipitation", "snowfall",
    "cloud_cover", "surface_pressure", "wind_speed_10m", "wind_gusts_10m",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
]
UNKNOWN_REALS = ["demand", "net_generation", "total_interchange"] + \
                [f"act_{v}" for v in WEATHER_VARS]
KNOWN_REALS = [f"fc_{v}" for v in WEATHER_VARS] + \
              ["fx_app_roll72", "fx_cdh_24h", "fx_hot_streak_day",
               "fx_night_min_app_prev", "time_idx"]
KNOWN_CATS = ["hour", "day_of_week", "day_of_month", "month", "is_holiday"]
CDH_BASE_C, HOT_DAY_APPARENT_C = 22.0, 24.0


def recompute_fx(df):
    """Identical logic to the dataset builder, on current fc_ columns."""
    app = df["fc_apparent_temperature"]
    df["fx_app_roll72"] = app.rolling(72, min_periods=24).mean()
    df["fx_cdh_24h"] = (app - CDH_BASE_C).clip(lower=0).rolling(24, min_periods=12).sum()

    d = pd.DataFrame({"date": df["ts"].dt.date, "app": app.values,
                      "hour": df["ts"].dt.hour})
    daily_max = d.groupby("date")["app"].max()
    is_hot = daily_max >= HOT_DAY_APPARENT_C
    streak, out = 0, {}
    for date, hot in is_hot.items():
        streak = streak + 1 if hot else 0
        out[date] = streak
    df["fx_hot_streak_day"] = d["date"].map(out).values

    night = d[d["hour"].isin([2, 3, 4, 5])].groupby("date")["app"].min()
    df["fx_night_min_app_prev"] = d["date"].map(night.shift(1)).values

    fx = ["fx_app_roll72", "fx_cdh_24h", "fx_hot_streak_day", "fx_night_min_app_prev"]
    df[fx] = df[fx].ffill().bfill()
    return df


def load_base():
    df = pd.read_csv(CSV_CLEAN, parse_dates=["ts"])
    for c in KNOWN_CATS:
        df[c] = df[c].astype(str).astype("category")
    df["series"] = df["series"].astype(str)
    return df.sort_values("time_idx").reset_index(drop=True)


def swap_lead(df, leads, d):
    """Return a copy of df whose fc_ columns hold previous_day{d} forecasts."""
    out = df.copy()
    sub = leads[["ts"] + [f"{v}_previous_day{d}" for v in WEATHER_VARS]].copy()
    sub.columns = ["ts"] + [f"new_{v}" for v in WEATHER_VARS]
    out = out.merge(sub, on="ts", how="left")
    for v in WEATHER_VARS:
        newv = out[f"new_{v}"]
        # small gaps: interpolate; remaining: fall back to existing lead-5 value
        newv = newv.interpolate(limit=6)
        out[f"fc_{v}"] = newv.fillna(out[f"fc_{v}"])
        out.drop(columns=[f"new_{v}"], inplace=True)
    return recompute_fx(out)


def build_training_ds(df):
    val_start_idx = int(df.loc[df["ts"] >= VAL_START, "time_idx"].min())
    train_df = df[df["time_idx"] < val_start_idx]
    return TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["series"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        target_normalizer=GroupNormalizer(groups=["series"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )


def predict_pass(model, training_ds, df, test_start_idx):
    test = TimeSeriesDataSet.from_dataset(
        training_ds, df, min_prediction_idx=test_start_idx,
        stop_randomization=True)
    dl = test.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    preds = model.predict(dl, mode="prediction", return_y=True,
                          return_index=True,
                          trainer_kwargs={"accelerator": "auto",
                                          "enable_progress_bar": True})
    y_hat = preds.output.cpu().numpy()
    y_true = preds.y[0].cpu().numpy()
    first_tidx = preds.index["time_idx"].values  # first decoder step per sample
    return y_hat, y_true, first_tidx


def main():
    pl.seed_everything(42)
    df = load_base()
    leads = pd.read_csv(CSV_LEADS, parse_dates=["ts"])
    test_start_idx = int(df.loc[df["split"] == "test", "time_idx"].min())

    training_ds = build_training_ds(df)
    model = TemporalFusionTransformer.load_from_checkpoint(CKPT)

    # month lookup by time_idx for the seasonal breakdown
    month_by_tidx = df.set_index("time_idx")["ts"].dt.month
    month_arr = np.full(int(df["time_idx"].max()) + DECODER_LEN + 2, -1)
    month_arr[month_by_tidx.index.values] = month_by_tidx.values

    day_mape, composite = {}, []
    for d in range(1, 6):
        print(f"\n=== Lead {d}: swapping fc_* to previous_day{d}, predicting ===")
        df_d = swap_lead(df, leads, d)
        y_hat, y_true, first_tidx = predict_pass(model, training_ds, df_d,
                                                 test_start_idx)
        s, e = (d - 1) * 24, d * 24
        yh, yt = y_hat[:, s:e], y_true[:, s:e]
        ape = np.abs(yt - yh) / np.clip(np.abs(yt), 1e-6, None)
        day_mape[d] = float(ape.mean() * 100)
        print(f"Day {d} lead-matched MAPE: {day_mape[d]:.2f}%  "
              f"(lead-5 run had ~{[4.36,4.49,4.51,4.52,4.52][d-1]:.2f}%, "
              f"baseline {BASELINE[d-1]:.2f}%)")

        # element-wise months for this day slice
        tidx_matrix = first_tidx[:, None] + np.arange(s, e)[None, :]
        composite.append((ape, month_arr[tidx_matrix]))

    # ---------------- summary ----------------
    all_ape = np.concatenate([a.ravel() for a, _ in composite])
    all_mon = np.concatenate([m.ravel() for _, m in composite])
    overall = float(all_ape.mean() * 100)

    season_map = {12: "Winter", 1: "Winter", 2: "Winter",
                  3: "Spring", 4: "Spring", 5: "Spring",
                  6: "Summer", 7: "Summer", 8: "Summer",
                  9: "Fall", 10: "Fall", 11: "Fall"}
    seasons = {}
    for season in ["Winter", "Spring", "Summer", "Fall"]:
        mask = np.isin(all_mon, [m for m, s2 in season_map.items() if s2 == season])
        seasons[season] = float(all_ape[mask].mean() * 100) if mask.any() else None

    print("\n================ LEAD-MATCHED RESULTS ================")
    for d in range(1, 6):
        print(f"Day {d}: {day_mape[d]:.2f}%   (baseline {BASELINE[d-1]:.2f}%)")
    print(f"Overall lead-matched MAPE: {overall:.2f}%")
    print("\nSeasonal breakdown (lead-matched composite):")
    for s2, v in seasons.items():
        marker = "  <- was 5.2% in the old model" if s2 == "Summer" else ""
        print(f"  {s2}: {v:.2f}%{marker}")

    out = {"day_mape": {str(k): round(v, 3) for k, v in day_mape.items()},
           "overall_mape": round(overall, 3),
           "seasonal_mape": {k: (round(v, 3) if v else None)
                             for k, v in seasons.items()},
           "checkpoint": CKPT}
    Path(OUT_JSON).write_text(json.dumps(out, indent=2))
    print(f"\nSaved -> {OUT_JSON}")


if __name__ == "__main__":
    main()

Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80/80 0:01:24 • 0:00:00 0.96it/s

Day 5 lead-matched MAPE: 4.52%  (lead-5 run had ~4.52%, baseline 4.16%)

================ LEAD-MATCHED RESULTS ================
Day 1: 3.51%   (baseline 3.84%)
Day 2: 3.69%   (baseline 4.07%)
Day 3: 3.92%   (baseline 4.12%)
Day 4: 4.13%   (baseline 4.14%)
Day 5: 4.52%   (baseline 4.16%)
Overall lead-matched MAPE: 3.96%

Seasonal breakdown (lead-matched composite):
  Winter: 3.37%
  Spring: 4.10%
  Summer: 4.85%  <- was 5.2% in the old model
  Fall: 3.54%

Saved -> metrics_lead_matched.json


In [2]:
%pip install pytorch-forecasting lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/848.6 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 13.7 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 119.1 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 325.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
